# CMA-MESO 1KM

CMA-MESO 基础 GRIB2 数据文件有 402 个要素场。

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

from reki.data_finder import find_local_file
from reki.format.grib.eccodes import load_field_from_file

In [2]:
start_time = pd.Timestamp.utcnow().floor(freq="D") - pd.Timedelta(days=2)
start_time_label = start_time.strftime("%Y%m%d%H")
forecast_time_label = "24h"
forecast_time = pd.to_timedelta(forecast_time_label)

meso_grib2_orig_file_path = find_local_file(
    "cma_meso_1km/grib2/orig",
    start_time=start_time,
    forecast_time=forecast_time,
)
meso_grib2_orig_file_path

PosixPath('/g3/COMMONDATA/OPER/CEMC/MESO_1KM/Prod-grib/2025081600/ORIG/rmf.hgra.2025081600024.grb2')

## 要素列表

使用 ecCodes 打印 CMA-GFS 基础 GRIB2 数据的要素列表。

In [3]:
from IPython.utils.capture import capture_output

with capture_output() as captured:
    !module load eccodes/2.29.0/intel && grib_ls -P count "{meso_grib2_orig_file_path}" 2>/dev/null

print(captured.stdout)

/g3/COMMONDATA/OPER/CEMC/MESO_1KM/Prod-grib/2025081600/ORIG/rmf.hgra.2025081600024.grb2
count        edition      centre       date         dataType     gridType     typeOfLevel  level        stepRange    shortName    packingType  
1            2            babj         20250816     fc           regular_ll   surface      0            0-24         unknown      grid_jpeg   
2            2            babj         20250816     fc           regular_ll   surface      0            0-24         asnow        grid_jpeg   
3            2            babj         20250816     fc           regular_ll   surface      0            24           t            grid_jpeg   
4            2            babj         20250816     fc           regular_ll   surface      0            0-24         str          grid_jpeg   
5            2            babj         20250816     fc           regular_ll   surface      0            0-24         ssr          grid_jpeg   
6            2            babj         20250816     f

使用 wgrib2 打印 CMA-GFS 基础 GRIB2 数据的要素列表。

In [4]:
from IPython.utils.capture import capture_output

with capture_output() as captured:
    !module load wgrib2/3.1.1/intel && wgrib2 "{meso_grib2_orig_file_path}" 2>/dev/null

print(captured.stdout)

1:0:d=2025081600:APCP:surface:0-1 day acc fcst:
2:44629570:d=2025081600:ASNOW:surface:0-1 day acc fcst:
3:59746327:d=2025081600:TMP:surface:1440 min fcst:
4:84489091:d=2025081600:NLWRF:surface:0-1 day acc fcst:
5:125346208:d=2025081600:NSWRF:surface:0-1 day acc fcst:
6:153828511:d=2025081600:HGT:surface:1440 min fcst:
7:170523817:d=2025081600:SPFH:2 m above ground:1440 min fcst:
8:191405145:d=2025081600:TMP:2 m above ground:1440 min fcst:
9:215037146:d=2025081600:UGRD:10 m above ground:1440 min fcst:
10:251580288:d=2025081600:VGRD:10 m above ground:1440 min fcst:
11:287308251:d=2025081600:TCDC:atmos col:1440 min fcst:
12:325826768:d=2025081600:LCDC:atmos col:1440 min fcst:
13:367562013:d=2025081600:MCDC:atmos col:1440 min fcst:
14:396349404:d=2025081600:HCDC:atmos col:1440 min fcst:
15:414772860:d=2025081600:TCIWV:entire atmosphere:1440 min fcst:
16:439284372:d=2025081600:TCOLI:entire atmosphere:1440 min fcst:
17:458043713:d=2025081600:HPBL:surface:1440 min fcst:
18:496909035:d=2025081

对 wgrib2 输出信息进行分组展示。

```{note}
本节代码由 ChatGPT 生成。
```

In [5]:
import re
import textwrap
from collections import defaultdict
from IPython.utils.capture import capture_output

with capture_output() as captured:
    !module load wgrib2/3.1.1/intel && wgrib2 "{meso_grib2_orig_file_path}" 2>/dev/null

text = captured.stdout

lines = text.strip().splitlines()
grouped = defaultdict(list)

pattern = re.compile(r'^(\d+):\d+:d=\d+:([^:]+):([^:]+):([^:]+):?$')

for line in lines:
    match = pattern.match(line)
    if match:
        index, param, level, fcst = match.groups()
        grouped[(param.strip(), fcst.strip())].append((level.strip(), int(index)))

WRAP_WIDTH = 80

for (param, fcst), level_index_list in grouped.items():
    level_index_list.sort(key=lambda x: x[1])
    level_strs = [f"{level} ({index})" for level, index in level_index_list]
    full_line = ', '.join(level_strs)
    wrapped = textwrap.fill(full_line, width=WRAP_WIDTH, subsequent_indent='  ', initial_indent='  ')
    print(f"{param} @ {fcst}:\n{wrapped}\n")

APCP @ 0-1 day acc fcst:
  surface (1)

ASNOW @ 0-1 day acc fcst:
  surface (2)

TMP @ 1440 min fcst:
  surface (3), 2 m above ground (8), 1000 mb (70), 975 mb (71), 950 mb (72), 925
  mb (73), 900 mb (74), 850 mb (75), 800 mb (76), 750 mb (77), 700 mb (78), 650
  mb (79), 600 mb (80), 550 mb (81), 500 mb (82), 450 mb (83), 400 mb (84), 350
  mb (85), 300 mb (86), 250 mb (87), 200 mb (88), 150 mb (89), 100 mb (90)

NLWRF @ 0-1 day acc fcst:
  surface (4)

NSWRF @ 0-1 day acc fcst:
  surface (5)

HGT @ 1440 min fcst:
  surface (6), 1000 mb (49), 975 mb (50), 950 mb (51), 925 mb (52), 900 mb (53),
  850 mb (54), 800 mb (55), 750 mb (56), 700 mb (57), 650 mb (58), 600 mb (59),
  550 mb (60), 500 mb (61), 450 mb (62), 400 mb (63), 350 mb (64), 300 mb (65),
  250 mb (66), 200 mb (67), 150 mb (68), 100 mb (69)

SPFH @ 1440 min fcst:
  2 m above ground (7), 1000 mb (154), 975 mb (155), 950 mb (156), 925 mb (157),
  900 mb (158), 850 mb (159), 800 mb (160), 750 mb (161), 700 mb (162), 650 mb
 